# Teste em Tempo Real (Webcam) — LIBRAS YOLO

Notebook para testar o modelo treinado em **tempo real** usando a webcam.

## Pré-requisitos
- Rodar **localmente** (Colab não acessa webcam local diretamente)
- Ter o `best.pt` baixado do Colab e colocado em `models/best.pt` (ou ajustar o caminho)
- Webcam funcional (interna ou USB)

## Ambiente recomendado: venv (economiza espaço e isola as dependências)

Antes de abrir este notebook, crie um ambiente virtual no **PowerShell do Windows** (Python 3.12):

```powershell
# 1. Vá até a pasta onde está este notebook + a pasta models/
# 2. Crie o venv
py -3.12 -m venv .venv-webcam
# 3. Ative o venv
.venv-webcam\Scripts\Activate.ps1
# 4. (opcional) atualize o pip
python -m pip install --upgrade pip
```

Depois, no VS Code/Jupyter, **selecione o kernel `.venv-webcam`** e rode a célula de instalação abaixo.
Para apagar tudo depois, basta deletar a pasta `.venv-webcam`.

## Controles (na janela da webcam)
| Tecla | Ação |
|---|---|
| `Q` | Sair |
| `S` | Salvar snapshot anotado em `real_world_test/images/` |
| `+` / `-` | Aumentar / diminuir threshold de confiança |

---
## 1. Instalação

In [1]:
# Instala versao CPU do PyTorch (muito menor que a CUDA) + ultralytics + opencv.
# A inferencia da webcam roda tranquilamente em CPU.
%pip install opencv-python
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
%pip install ultralytics

Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached torch-2.12.0%2Bcpu-cp312-cp312-win_amd64.whl.metadata (32 kB)
  Using cached torchvision-0.27.0%2Bcpu-cp312-cp312-win_amd64.whl.metadata (5.6 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached setuptools-70.2.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached pillow-12.2.0-cp312-cp312-win_amd64.whl.metadata (9.0 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-win_amd64.whl.metadata (2.8 kB)
   ---------------------------------------- 0.0/122.9 MB ? eta -:--:--
   ---------------------------------------- 

In [2]:
import os
import time
from datetime import datetime
import cv2
from ultralytics import YOLO

print(f'OpenCV: {cv2.__version__}')

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\eduar\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
OpenCV: 4.13.0


---
## 2. Configuração

Ajuste o caminho do modelo treinado e o índice da webcam se precisar.

In [3]:
# Caminho do modelo treinado (best.pt baixado do Colab)
MODEL_PATH = 'models/best.pt'

# Pasta para salvar snapshots
SNAPSHOT_DIR = 'real_world_test/images'

# Índice da webcam (0 = padrão, 1 = segunda câmera, etc.)
CAMERA_INDEX = 0

# Resolução desejada (a webcam pode ajustar para a mais próxima suportada)
FRAME_WIDTH = 1280
FRAME_HEIGHT = 720

# Threshold inicial de confiança (0–1)
CONF_THRESHOLD = 0.35

# Imagem de input para o YOLO (tem que bater com imgsz do treino)
INFER_IMGSZ = 640

assert os.path.exists(MODEL_PATH), f'Modelo não encontrado em: {MODEL_PATH}'
os.makedirs(SNAPSHOT_DIR, exist_ok=True)
print(f'✅ Modelo: {MODEL_PATH}')
print(f'✅ Snapshots vão para: {SNAPSHOT_DIR}')

✅ Modelo: models/best.pt
✅ Snapshots vão para: real_world_test/images


In [4]:
# Carrega o modelo (apenas uma vez)
model = YOLO(MODEL_PATH)
print(f'Classes do modelo: {model.names}')

Classes do modelo: {0: 'A', 1: 'Ajuda', 2: 'B', 3: 'Banheiro', 4: 'C', 5: 'Cade', 6: 'Casa', 7: 'D', 8: 'E', 9: 'Em pe', 10: 'Eu', 11: 'F', 12: 'Febre', 13: 'G', 14: 'Gosto', 15: 'I', 16: 'L', 17: 'M', 18: 'N', 19: 'O', 20: 'P', 21: 'Policial', 22: 'Q', 23: 'R', 24: 'S', 25: 'T', 26: 'Te amo', 27: 'Telefone', 28: 'Tenho', 29: 'U', 30: 'V', 31: 'Vacina', 32: 'Voce', 33: 'W', 34: 'Y'}


---
## 3. Loop em Tempo Real

Abre uma janela do OpenCV com inferência contínua. Use `Q` para sair.

> ⚠️ Se a janela ficar congelada ou não responder ao `Q`, clique nela primeiro para focar e tente de novo. Em último caso, **interrompa o kernel** (botão de parar).

In [9]:
cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)

if not cap.isOpened():
    raise RuntimeError(f'Não consegui abrir a câmera no índice {CAMERA_INDEX}.')

conf_atual = CONF_THRESHOLD
window_name = 'LIBRAS YOLO — Q: sair | S: snapshot | +/-: confianca'

# Buffer para média móvel de FPS
fps_buffer = []
buffer_size = 30

try:
    while True:
        t0 = time.time()
        ret, frame = cap.read()
        if not ret:
            print('Erro ao ler frame.')
            break

        # Inferência
        result = model.predict(frame, conf=conf_atual, imgsz=INFER_IMGSZ, verbose=False)[0]
        annotated = result.plot()

        # FPS
        dt = time.time() - t0
        fps_buffer.append(1.0 / max(dt, 1e-6))
        if len(fps_buffer) > buffer_size:
            fps_buffer.pop(0)
        fps = sum(fps_buffer) / len(fps_buffer)

        # Overlay de status (FPS + threshold + número de detecções)
        overlay = annotated.copy()
        cv2.rectangle(overlay, (5, 5), (380, 95), (0, 0, 0), -1)
        annotated = cv2.addWeighted(overlay, 0.6, annotated, 0.4, 0)
        cv2.putText(annotated, f'FPS: {fps:5.1f}', (15, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.putText(annotated, f'Conf: {conf_atual:.2f}  Det: {len(result.boxes)}', (15, 75),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        cv2.imshow(window_name, annotated)

        # Captura de teclado
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:  # Q ou ESC
            break
        elif key == ord('s'):
            ts = datetime.now().strftime('%Y%m%d_%H%M%S')
            path = os.path.join(SNAPSHOT_DIR, f'snapshot_{ts}.jpg')
            cv2.imwrite(path, annotated)
            print(f'📸 Snapshot salvo: {path}')
        elif key in (ord('+'), ord('=')):
            conf_atual = min(0.95, conf_atual + 0.05)
            print(f'Confiança: {conf_atual:.2f}')
        elif key == ord('-'):
            conf_atual = max(0.05, conf_atual - 0.05)
            print(f'Confiança: {conf_atual:.2f}')
finally:
    cap.release()
    cv2.destroyAllWindows()
    # Workaround para janelas presas no Windows/Jupyter
    for _ in range(4):
        cv2.waitKey(1)
    print('🛑 Webcam liberada.')

📸 Snapshot salvo: real_world_test/images\snapshot_20260606_155515.jpg
📸 Snapshot salvo: real_world_test/images\snapshot_20260606_155541.jpg
📸 Snapshot salvo: real_world_test/images\snapshot_20260606_155609.jpg
🛑 Webcam liberada.
